In [4]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup

In [ ]:
def fetch_banapresso():
    url = "https://www.banapresso.com/"

    driver = webdriver.Chrome()
    driver.maximize_window()

    driver.get(url)
    time.sleep(2)

    action = ActionChains(driver)

    first_tag = driver.find_element(
        By.CSS_SELECTOR,
        "#wrap > header > div > ul > li:nth-child(2) > a > span"
    )

    second_tag = driver.find_element(
        By.CSS_SELECTOR,
        "#wrap > header > div > ul > li:nth-child(2) > ul > li:nth-child(1) > a > span"
    )

    action.move_to_element(first_tag).move_to_element(second_tag).click().perform()
    time.sleep(3)

    try:
        popup_tag = driver.find_element(
            By.CSS_SELECTOR,
            "#root > div.sc-ff716c63-0.fOAigp > div > div.p_btm > button"
        )

        action.move_to_element(popup_tag).click().perform()

    except:
        print("팝업이 없거나 이미 닫혀 있습니다.")


    before_count = 0

    while True:
        store_names = driver.find_elements(
            By.CSS_SELECTOR,
            ".store_name_map .name"
        )

        current_count = len(store_names)
        # print(f"현재 로딩된 매장 수: {current_count}")

        if current_count == before_count:
            break

        before_count = current_count

        driver.execute_script(
            """
            const listBox = document.querySelector('.store_shop_list');
            if (listBox) {
                listBox.scrollTop = listBox.scrollHeight;
            }
            """
        )

        time.sleep(1)

    req = driver.page_source
    soup = BeautifulSoup(req, "html.parser")
    
    stores = soup.select(".store_name_map")
    store_data = []

    for store in stores:
        name_tag = store.select_one(".name")
        address_tag = store.select_one(".address")
        time_tag = store.select_one(".store-time-wrap")
        parking_tag = store.select_one(".parking")

        store_name = name_tag.get_text(strip=True) if name_tag else ""
        store_address = address_tag.get_text(strip=True) if address_tag else ""
        store_time = time_tag.get_text(" ", strip=True) if time_tag else ""
        parking_list = parking_tag.get_text(strip=True) if parking_tag else ""

        if store_name and store_address:
            store_data.append({
                "매장명": store_name,
                "주소": store_address,
                "영업정보": store_time,
                '주차 정보' : parking_list
            })

    df = pd.DataFrame(store_data)
    df.index = df.index + 1

    driver.quit()
    
    return df


banapresso_df = fetch_banapresso()

banapresso_df.to_csv(
    "banapresso.csv",
    index=False,
    encoding='utf-8-sig'
)

banapresso_df


,매장명,주소,영업정보,주차 정보
1,가락몰점,"서울특별시 송파구 양재대로 932, 업무동 1층 로비",OPEN 07:30~23:30,
2,가산디지털단지역점,서울시 금천구 가산동 60-3,CLOSE 07:00~19:00,
3,가산안양천점,"서울 금천구 가산 디지털2로 127-143, 101호",CLOSE 07:00~20:00,
4,가산어반워크점,"서울시 금천구 가산디지털2로 135, 1동 142호",CLOSE 07:00~17:30,
5,가산우림라이온스점,서울 금천구 가산디지털1로 168 b131호,CLOSE 07:00~17:00,
...,...,...,...,...
223,홍대입구역사거리점,서울 마포구 양화로 129,OPEN 07:00~21:00 주차불가,주차불가
224,회기역사거리점,서울 동대문구 회기로 176 (회기동81),OPEN 07:00~22:00,
225,AK금정점,경기도 군포시 금정동 689번지 AK플라자 금정점 2층,OPEN 07:30~22:30 지하 주차장 이용 가능,지하 주차장 이용 가능
226,가산에이스비즈포레점,서울 금천구 가산동 459-23,CLOSE,
